# OS-A01-A06 — National Information Ecosystem Laboratory

**Pathway:** SRAI Official Statistics and AI  
**Module:** OS-A01 — The National Information Ecosystem  
**Laboratory:** Kintaba synthetic case  
**Version:** 0.1 candidate

This laboratory turns institutional evidence into an auditable ecosystem map and an initial AI-opportunity screen. It does **not** treat a high model score as proof that a system is fit for official statistics.

## Learning contract

By the end, you should be able to:

1. distinguish actors, mandates, assets, flows, decision needs and risks;
2. identify gaps in coordination, timeliness, coverage and interoperability;
3. separate promising AI opportunities from deployment-ready use cases;
4. export a reproducible evidence package.

All Kintaba data are synthetic. Do not paste confidential or personally identifiable records into this notebook.

In [ ]:
from pathlib import Path
import json, platform, sys
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt

def locate_data_dir():
    candidates = [Path("../data"), Path("data"), Path.cwd().parent / "data"]
    for candidate in candidates:
        if (candidate / "kintaba_actors.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Kintaba data folder. Run from notebook/ or package root.")

DATA_DIR = locate_data_dir()
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
RUN_UTC = datetime.now(timezone.utc).isoformat()
print({"python": sys.version.split()[0], "pandas": pd.__version__, "platform": platform.platform(), "data_dir": str(DATA_DIR), "run_utc": RUN_UTC})

## 1. Load the evidence tables

Each table has a narrow purpose. Keeping them separate prevents institutional claims, data quality measurements and policy needs from being blended without traceability.

In [ ]:
files = {
    "actors": "kintaba_actors.csv", "assets": "kintaba_assets.csv",
    "flows": "kintaba_flows.csv", "needs": "kintaba_needs.csv",
    "evidence": "kintaba_evidence.csv", "risks": "kintaba_risks.csv",
    "opportunities": "kintaba_ai_opportunities.csv",
}
tables = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
for name, frame in tables.items():
    print(f"{name:14s} {frame.shape[0]:2d} rows × {frame.shape[1]:2d} columns")

In [ ]:
actors, assets, flows = tables["actors"], tables["assets"], tables["flows"]
needs, evidence = tables["needs"], tables["evidence"]
risks, opportunities = tables["risks"], tables["opportunities"]

assert actors["actor_id"].is_unique and actors["actor_id"].notna().all()
assert assets["asset_id"].is_unique and assets["custodian_id"].isin(actors["actor_id"]).all()
assert flows["source_actor_id"].isin(actors["actor_id"]).all()
assert flows["destination_actor_id"].isin(actors["actor_id"]).all()
assert flows["asset_id"].isin(assets["asset_id"]).all()
assert needs["decision_owner_id"].isin(actors["actor_id"]).all()
print("Referential-integrity checks: PASS")

## 2. Actors, authority and coordination

An actor's existence is not evidence of an effective coordination relationship. Inspect the mandate and the observed coordination score together.

In [ ]:
actor_view = actors[["actor_id", "actor_name", "actor_type", "primary_role", "coordination_score"]].sort_values(["coordination_score", "actor_id"])
actor_view

In [ ]:
ax = actor_view.set_index("actor_id")["coordination_score"].sort_values().plot.barh(figsize=(9, 5), color="#1f6f8b")
ax.set(title="Kintaba actor coordination score", xlabel="Score (1=weak, 5=strong)", ylabel="Actor")
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actor_coordination_scores.png", dpi=160)
plt.show()

## 3. Statistical assets and fitness

Completeness alone is not fitness for use. Timeliness, coverage, access, concepts, linkage and operational continuity also matter.

In [ ]:
asset_view = assets.merge(actors[["actor_id", "actor_name"]], left_on="custodian_id", right_on="actor_id", how="left")
asset_view[["asset_id", "asset_name", "actor_name", "frequency", "timeliness_days", "completeness_pct", "fitness_score"]].sort_values("fitness_score")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(assets["timeliness_days"], assets["completeness_pct"], s=assets["fitness_score"]*70, c=assets["fitness_score"], cmap="viridis", alpha=.8)
for row in assets.itertuples():
    ax.annotate(row.asset_id, (row.timeliness_days, row.completeness_pct), xytext=(4,4), textcoords="offset points", fontsize=8)
ax.set(title="Asset timeliness, completeness and fitness", xlabel="Timeliness delay (days; lower is better)", ylabel="Completeness (%)")
ax.grid(alpha=.25)
plt.colorbar(scatter, ax=ax, label="Fitness score")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "asset_fitness.png", dpi=160)
plt.show()

## 4. Trace the information flows

The flow table makes exchange mechanisms, formal agreements and failure risks explicit. A flow is not reliable merely because it occurs.

In [ ]:
names = actors.set_index("actor_id")["actor_name"].to_dict()
flow_view = flows.copy()
flow_view["source"] = flow_view["source_actor_id"].map(names)
flow_view["destination"] = flow_view["destination_actor_id"].map(names)
flow_view[["flow_id", "source", "destination", "asset_id", "frequency", "formal_agreement", "risk"]]

In [ ]:
flow_summary = flows.groupby(["source_actor_id", "destination_actor_id"], as_index=False).size().sort_values("size", ascending=False)
flow_summary.rename(columns={"size":"flow_count"})

## 5. Decision needs and evidence fitness

Start with the decision and its deadline—not with an algorithm. The highest-priority needs reveal where timeliness and provenance failures have public consequences.

In [ ]:
needs.sort_values(["priority", "need_id"], ascending=[False, True])

In [ ]:
evidence[["evidence_id", "claim", "implication", "quality"]]

## 6. Risk matrix

Risk exposure is a screening signal, not a substitute for expert review. Likelihood × impact is used here only to prioritize discussion.

In [ ]:
risks = risks.copy()
risks["exposure"] = risks["likelihood"] * risks["impact"]
risks.sort_values("exposure", ascending=False)

In [ ]:
ax = risks.sort_values("exposure").plot.barh(x="risk_id", y="exposure", figsize=(9, 4.5), legend=False, color="#c65d3b")
ax.set(title="Risk exposure screen", xlabel="Likelihood × impact", ylabel="Risk")
ax.axvline(16, color="#7a1f1f", linestyle="--", linewidth=1)
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "risk_exposure.png", dpi=160)
plt.show()

## 7. AI-opportunity screen

The score rewards public value, data readiness and operational readiness, while penalizing methodological risk. It is deliberately conservative: a ranked opportunity still requires legal, statistical, ethical and operational assurance.

In [ ]:
opportunities = opportunities.copy()
opportunities["screen_score"] = (
    0.40 * opportunities["public_value"] +
    0.30 * opportunities["data_readiness"] +
    0.20 * opportunities["operational_readiness"] -
    0.30 * opportunities["methodological_risk"]
).round(2)
opportunities.sort_values("screen_score", ascending=False)

In [ ]:
ax = opportunities.sort_values("screen_score").plot.barh(x="opportunity_id", y="screen_score", figsize=(9, 4.5), legend=False, color="#5b4b8a")
ax.set(title="Conservative AI-opportunity screen", xlabel="Screen score (not an approval score)", ylabel="Opportunity")
ax.axvline(0, color="black", linewidth=.8)
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ai_opportunity_screen.png", dpi=160)
plt.show()

## 8. Learner challenge

Choose one opportunity and write a one-page pilot proposition answering:

- What decision will it support, for whom, and by when?
- What is the target statistical population and information boundary?
- Which sources are authoritative, experimental or incomplete?
- What baseline must the AI method outperform?
- How will coverage error, linkage error and uncertainty be measured?
- What human approval and revision controls remain mandatory?
- What evidence would trigger **stop**, **revise** or **scale**?

Do not change the synthetic evidence to make a preferred solution appear ready.

## 9. Export the evidence package

This final cell exports machine-readable summaries and a compact audit record. These are learning outputs, not official statistics.

In [ ]:
ranked = opportunities.sort_values("screen_score", ascending=False)
ranked.to_csv(OUTPUT_DIR / "ai_opportunity_screen.csv", index=False)
risks.sort_values("exposure", ascending=False).to_csv(OUTPUT_DIR / "risk_register_prioritized.csv", index=False)

run_record = {
    "laboratory_id": "OS-A01-A06",
    "module_id": "OS-A01",
    "case": "Kintaba synthetic national information ecosystem",
    "run_utc": RUN_UTC,
    "source_files": files,
    "row_counts": {name: int(len(frame)) for name, frame in tables.items()},
    "checks": {"referential_integrity": "PASS", "synthetic_data_only": True},
    "warning": "Screen scores support discussion and do not authorize deployment or statistical release.",
}
(OUTPUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print(f"Evidence package written to {OUTPUT_DIR.resolve()}")
print("Laboratory validation assertions: PASS")